# Lab 7: Classification

<img src="ds4all-logo.png" alt="Data Science for All Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

Welcome to Lab 7 of *Data Science for All*! This is the last lab of the semester.

In lecture you classified patients with the $k$-nearest neighbor classifier: to predict the class of a
new point, find the $k$ points in the training set that are closest to it, and take a majority vote of
their classes. In this lab you'll build nearest neighbor classifiers for two new data sets: banknotes
that are legitimate or fraudulent, and penguins of three species.

## Today's lab

In today's lab, you'll learn how to:

1. compute the distance between two rows of a table;
2. split a data set into a training set and a test set;
3. classify the test set with `knn()` and measure the classifier's accuracy; and
4. explain why accuracy on the training set is too optimistic.

## Resources

Keep these open while you work:

- **Reading:** [Classification](https://dats.columbian.gwu.edu/ds4all/modules/classification.html), adapted from Chapter 17 of the Data 8 textbook

These are the tools you'll use most:

| If you want to... | Use |
|---|---|
| keep some columns of a tibble | `select()` |
| keep rows by their position | `slice()` |
| add columns to a tibble | `mutate()` |
| pick a value row by row, based on a condition | `if_else()` |
| classify points by their nearest neighbors | `knn(train = ..., test = ..., cl = ..., k = ...)` |
| find the proportion of predictions that are right | `mean(predicted == actual)` |

In [2]:
# Initialize the autograder and JupyterHub helper
source("helpers.R")

library(readr)
library(dplyr)
library(ggplot2)
library(class)

options(repr.matrix.max.rows = 20)
options(repr.plot.width = 7, repr.plot.height = 4.5)

The `class` package is new. It comes with R, and it has one function we need: `knn()`, which finds
the nearest neighbors and takes the majority vote in one step. The notes do the same job with
`nearest_neighbor()` from tidymodels, which needs a package our JupyterHub doesn't have.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 1. Is this banknote real?

Researchers photographed 1,372 banknotes. Some were legitimate and some were fraudulent. From each
photograph they computed four numbers that describe the image. You saw scatter plots of these
banknotes in the notes. Now you'll build the classifier.

Each row of `banknotes` is one banknote. The four attributes are `WaveletVar`, `WaveletSkew`,
`WaveletCurt`, and `Entropy`. The `Class` column is 0 for a legitimate banknote and 1 for a fraudulent
one, so we turn it into a factor with labels.

In [3]:
banknotes <- read_csv("data/banknote.csv", show_col_types = FALSE) |>
  mutate(Class = factor(Class, levels = c(0, 1), labels = c("Legitimate", "Fraudulent")))

banknotes

WaveletVar,WaveletSkew,WaveletCurt,Entropy,Class
<dbl>,<dbl>,<dbl>,<dbl>,<fct>
3.62160,8.6661,-2.80730,-0.44699,Legitimate
4.54590,8.1674,-2.45860,-1.46210,Legitimate
3.86600,-2.6383,1.92420,0.10645,Legitimate
3.45660,9.5228,-4.01120,-3.59440,Legitimate
0.32924,-4.4552,4.57180,-0.98880,Legitimate
4.36840,9.6718,-3.96060,-3.16250,Legitimate
3.59120,3.0129,0.72888,0.56421,Legitimate
2.09220,-6.8100,8.46360,-0.60216,Legitimate
3.20320,5.7588,-0.75345,-0.61251,Legitimate


---

**Question 1.1.** Make a scatter plot of the banknotes with `WaveletVar` on the horizontal axis and
`WaveletCurt` on the vertical axis. Color each point by its `Class`. Name the plot `banknote_scatter`,
and replace each `...` in `labs()` with a title and axis labels in plain words.

In [ ]:
banknote_scatter <- ggplot(banknotes) +
  aes(x = ..., y = ..., color = ...) +
  geom_point(alpha = 0.5) +
  scale_color_manual(values = c("Legitimate" = "gold", "Fraudulent" = "darkblue")) +
  labs(
    title = "...",
    x = "...",
    y = "..."
  ) +
  theme_minimal(base_size = 16)

banknote_scatter

In [ ]:
. <- ottr::check("tests/q1_1.R")

The fraudulent banknotes are mostly on the left, but the two colors overlap in the middle. A
classifier that uses only these two attributes will make some mistakes there.

A nearest neighbor classifier needs to know how far apart two banknotes are. With four attributes,
each banknote is a point in four dimensions, and the distance between two points is

$$
D = \sqrt{(\text{difference in attribute 1})^2 + \dots + (\text{difference in attribute 4})^2}.
$$

---

**Question 1.2.** Fill in the function `distance()`. It takes two numeric vectors of the same length
and returns the distance between them: subtract, square, add up, and take the square root. Then use it
to find the distance between the first two banknotes, and store it in `first_two_distance`.

The cell already makes `first_banknote` and `second_banknote`, the attributes of rows 1 and 2 as
numeric vectors.

In [ ]:
distance <- function(point1, point2) {
  ...
}

banknote_attributes <- banknotes |> select(-Class)

first_banknote  <- banknote_attributes |> slice(1) |> as.numeric()
second_banknote <- banknote_attributes |> slice(2) |> as.numeric()

first_two_distance <- NULL # YOUR CODE HERE

first_two_distance

In [ ]:
. <- ottr::check("tests/q1_2.R")

Both of those banknotes are legitimate. The last banknote in the table is fraudulent, and it is much
farther from the first one:

In [ ]:
last_banknote <- banknote_attributes |> slice(1372) |> as.numeric()

distance(first_banknote, last_banknote)

To classify one new banknote, a nearest neighbor classifier computes this distance to every
banknote in the training set. `knn()` will do that for us.

### Training and testing

To find out how accurate a classifier is, we can't test it on the banknotes it learned from. So we
split the data: a **training set** to build the classifier, and a **test set**, kept aside, to measure
its accuracy.

The next cell shuffles the rows of `banknotes` at random.

**Leave `set.seed(7)` as it is, number included.** The seed makes R shuffle the rows the same way every
time, and the autograder expects that exact shuffle.

In [ ]:
set.seed(7)  # leave this line as it is: the autograder needs this exact shuffle

banknotes_shuffled <- banknotes |> slice_sample(prop = 1)

banknotes_shuffled

---

**Question 1.3.** There are 1,372 banknotes. Put the first 686 rows of `banknotes_shuffled` in
`banknote_train` and the remaining 686 rows in `banknote_test`.

In [ ]:
banknote_train <- NULL # YOUR CODE HERE
banknote_test  <- NULL # YOUR CODE HERE

nrow(banknote_train)
nrow(banknote_test)

In [ ]:
. <- ottr::check("tests/q1_3.R")

### Classifying with `knn()`

`knn()` takes four things:

- `train`: the attributes of the training set, with no class column;
- `test`: the attributes of the points to classify, with the same columns as `train`;
- `cl`: the classes of the training set, as a vector; and
- `k`: the number of neighbors that vote.

It returns one predicted class for each row of `test`. Here it classifies one made-up banknote with a
`WaveletVar` of 0 and a `WaveletCurt` of 2, using the two attributes from your scatter plot:

In [ ]:
new_banknote <- tibble(WaveletVar = 0, WaveletCurt = 2)

knn(
  train = banknote_train |> select(WaveletVar, WaveletCurt),
  test  = new_banknote,
  cl    = banknote_train$Class,
  k     = 5
)

Find the point (0, 2) on your scatter plot. Most of its neighbors are gold, so the prediction makes
sense.

---

**Question 1.4.** Classify every banknote in the test set with a 5-nearest neighbor classifier that
uses only `WaveletVar` and `WaveletCurt`.

1. `train_two` holds those two columns of the training set. Make `test_two`, the same two columns of
   the test set.
2. Fill in `knn()` and store the predictions in `predicted_two`.
3. Compute the classifier's accuracy: the proportion of banknotes in the test set whose predicted class
   matches their actual class. Store it in `accuracy_two`.

*Hint:* `predicted_two == banknote_test$Class` is `TRUE` for each correct prediction, and the `mean()`
of `TRUE`s and `FALSE`s is the proportion of `TRUE`s.

In [ ]:
train_two <- banknote_train |> select(WaveletVar, WaveletCurt)
test_two  <- NULL # YOUR CODE HERE

predicted_two <- knn(train = ..., test = ..., cl = ..., k = ...)

accuracy_two <- NULL # YOUR CODE HERE

accuracy_two

In [ ]:
. <- ottr::check("tests/q1_4.R")

The classifier gets about 9 out of 10 banknotes right. Its mistakes are in the region where the two
colors overlap:

In [ ]:
banknote_test |>
  mutate(result = if_else(predicted_two == Class, "correct", "wrong")) |>
  ggplot() +
  aes(x = WaveletVar, y = WaveletCurt, color = result) +
  geom_point(alpha = 0.5) +
  scale_color_manual(values = c("correct" = "gray60", "wrong" = "red")) +
  labs(
    title = "Test set: where the classifier is wrong",
    x = "WaveletVar",
    y = "WaveletCurt"
  ) +
  theme_minimal(base_size = 16)

---

**Question 1.5.** Now use all four attributes. Make `train_four` and `test_four`, classify the test
set with a 5-nearest neighbor classifier, and store the predictions in `predicted_four` and the
accuracy in `accuracy_four`.

*Hint:* `select(-Class)` keeps every column except `Class`.

In [ ]:
train_four <- NULL # YOUR CODE HERE
test_four  <- NULL # YOUR CODE HERE

predicted_four <- NULL # YOUR CODE HERE

accuracy_four <- NULL # YOUR CODE HERE

accuracy_four

Why is the classifier with four attributes more accurate?

- `A.` It learned from more banknotes, so each point in the test set had more neighbors nearby.
- `B.` With four attributes, `k = 5` lets more neighbors vote than it does with two attributes.
- `C.` The classes overlap in two attributes, and the other two attributes help tell them apart.
- `D.` The test set is easier with four attributes, because it then has fewer banknotes in it.

Set `attributes_answer` to the letter, as a string.

In [ ]:
attributes_answer <- NULL # YOUR CODE HERE

attributes_answer

In [ ]:
. <- ottr::check("tests/q1_5.R")

Both classifiers used the same training set, the same test set, and the same `k`. The only change
was the attributes. Banknotes that sit on top of each other in two dimensions can be far apart in four,
so each new attribute can give the classifier information it didn't have.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


## 2. Which penguin is this?

Every classifier so far has answered a yes or no question. Nearest neighbor classification also works
when there are more than two classes: the neighbors vote, and the class with the most votes wins.

The `penguins` data has measurements of 342 penguins near Palmer Station in Antarctica, from three
species: Adelie, Chinstrap, and Gentoo. We'll classify them with three measurements, all in
millimeters: `bill_length_mm`, `bill_depth_mm`, and `flipper_length_mm`.

*Data: Gorman, Williams, and Fraser (2014), from the `palmerpenguins` R package by Horst, Hill, and
Gorman.*

In [ ]:
penguins <- read_csv("data/penguins.csv", show_col_types = FALSE)

penguins

In [ ]:
ggplot(penguins) +
  aes(x = flipper_length_mm, y = bill_length_mm, color = species) +
  geom_point(alpha = 0.5) +
  scale_color_manual(values = c("Adelie" = "darkorange", "Chinstrap" = "purple", "Gentoo" = "darkcyan")) +
  labs(
    title = "Palmer penguins",
    x = "Flipper length (mm)",
    y = "Bill length (mm)"
  ) +
  theme_minimal(base_size = 16)

### Adelie or not?

We'll start with a yes or no question, as in the notes: is this penguin an Adelie?

---

**Question 2.1.** Make `penguins_grouped`: `penguins` with a new column, `group`, that is `"Adelie"`
for Adelie penguins and `"Other"` for the other two species.

In [ ]:
penguins_grouped <- penguins |>
  mutate(group = ...)

penguins_grouped |> count(species, group)

In [ ]:
. <- ottr::check("tests/q2_1.R")

The next cell shuffles the penguins and splits them in half, as you did in Question 1.3. It also
picks out the three attributes of each half.

**Leave `set.seed(7)` as it is here too.**

In [ ]:
set.seed(7)  # leave this line as it is: the autograder needs this exact shuffle

penguins_shuffled <- penguins_grouped |> slice_sample(prop = 1)

penguin_train <- penguins_shuffled |> slice(1:171)
penguin_test  <- penguins_shuffled |> slice(172:342)

penguin_train_attributes <- penguin_train |> select(bill_length_mm, bill_depth_mm, flipper_length_mm)
penguin_test_attributes  <- penguin_test |> select(bill_length_mm, bill_depth_mm, flipper_length_mm)

---

**Question 2.2.** Classify each penguin in the test set as `"Adelie"` or `"Other"` with a 5-nearest
neighbor classifier. Store the predictions in `group_predicted` and the accuracy in `group_accuracy`.

*Hint:* The classes of the training set are in `penguin_train$group`.

In [ ]:
group_predicted <- NULL # YOUR CODE HERE

group_accuracy <- NULL # YOUR CODE HERE

group_accuracy

In [ ]:
. <- ottr::check("tests/q2_2.R")

### Three species

To classify all three species, only one thing changes: the classes we give `knn()`.

---

**Question 2.3.** Classify each penguin in the test set as one of the three species with a 5-nearest
neighbor classifier. Store the predictions in `species_predicted` and the accuracy in
`species_accuracy`.

In [ ]:
species_predicted <- NULL # YOUR CODE HERE

species_accuracy <- NULL # YOUR CODE HERE

species_accuracy

In [ ]:
. <- ottr::check("tests/q2_3.R")

With three classes, the accuracy alone doesn't say which species get confused with each other. This
table counts each combination of actual and predicted species:

In [ ]:
penguin_test |>
  mutate(predicted = species_predicted) |>
  count(species, predicted)

The classifier never gets a Gentoo wrong. Its mistakes are almost all Chinstraps that it calls
Adelies. Telling three species apart is a little harder than picking out the Adelies, because the
classifier now also has to tell a Chinstrap from a Gentoo.

### Testing on the training set

---

**Question 2.4.** Use a **1**-nearest neighbor classifier for the three species, and compute its
accuracy two ways:

- `train_accuracy_k1`: classify the penguins in the **training** set, and compare with their actual
  species; and
- `test_accuracy_k1`: classify the penguins in the **test** set, and compare with their actual species.

Both classifiers learn from the training set. Only the `test` argument and the species you compare
with change.

In [ ]:
train_accuracy_k1 <- NULL # YOUR CODE HERE

test_accuracy_k1 <- NULL # YOUR CODE HERE

c(train_accuracy_k1, test_accuracy_k1)

Why is the accuracy on the training set so high?

- `A.` The training set was shuffled, so it is easier to classify than the test set is.
- `B.` Each penguin in the training set is its own nearest neighbor, so it gets its own species.
- `C.` With `k = 1` no vote can end in a tie, so the classifier can't make any mistakes.
- `D.` The three species don't overlap at all in these three measurements of the penguins.

Set `training_answer` to the letter, as a string.

In [ ]:
training_answer <- NULL # YOUR CODE HERE

training_answer

In [ ]:
. <- ottr::check("tests/q2_4.R")

An accuracy of 100% on the training set says nothing about new penguins. The accuracy on the test
set is the honest estimate, which is why we set the test set aside before building the classifier.

### One last thing: units matter

The data also has each penguin's body mass, in grams. More attributes helped with the banknotes. Watch
what happens here:

In [ ]:
train_with_mass <- penguin_train |> select(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)
test_with_mass  <- penguin_test |> select(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)

mass_predicted <- knn(train = train_with_mass, test = test_with_mass, cl = penguin_train$species, k = 5)

mean(mass_predicted == penguin_test$species)

The accuracy drops to about 80%. Body mass is in grams, so two penguins can differ by hundreds of
grams but only by a few millimeters. The distance is then almost all body mass, and the other three
attributes hardly count.

The fix is one you already know from Lab 6: put every attribute in standard units, so that each one is
measured in SDs from its average.

In [ ]:
standard_units <- function(numbers) {
  (numbers - mean(numbers)) / sd(numbers)
}

penguins_su <- penguins_shuffled |>
  mutate(
    bill_length_mm    = standard_units(bill_length_mm),
    bill_depth_mm     = standard_units(bill_depth_mm),
    flipper_length_mm = standard_units(flipper_length_mm),
    body_mass_g       = standard_units(body_mass_g)
  ) |>
  select(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)

su_predicted <- knn(train = penguins_su |> slice(1:171), test = penguins_su |> slice(172:342),
                    cl = penguin_train$species, k = 5)

mean(su_predicted == penguin_test$species)

In standard units, the four attributes count equally, and the accuracy is back up.

<br/><br/>

<hr style="border: 1px solid #fdb515;" />


You're done with Lab 7, the last lab of the semester!

Here's what you can now do: compute the distance between two rows, split data into a training set and
a test set, classify with `knn()` for two classes or more, measure accuracy on the test set, and
explain why accuracy on the training set is too optimistic.

**Important submission information:** Be sure to run the tests and verify that they all pass, then
choose **Save Notebook** from the **File** menu, then Download the .ipynb. Then, go to your course's
Gradescope and submit the corresponding assignment. The name of this assignment is "Lab 07".

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
# Re-run all the autograder tests in the tests/ folder.
. <- sapply(list.files("tests", pattern = "\\.R$", full.names = TRUE), function(f) ottr::check(f))

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all
images/graphs appear in the output. Click Download in the bar at the top of the notebook. **Please save
before downloading!** Submit your final version to Gradescope.